# Recommendation Modelling

Goal: recommend new Spotify tracks that are not already in the user's playlist.

This notebook builds a content-based recommender. It represents the user's playlist as a taste profile, represents candidate tracks with the same feature space, and ranks candidates by similarity to the playlist profile.

## 1. Setup

The recommender needs two datasets:

- `data/df.csv`: the user's existing playlist from the earlier pipeline.
- `data/candidate_tracks.csv`: songs that are not necessarily in the playlist and can be recommended.

If `candidate_tracks.csv` does not exist yet, the optional Spotify API section below can create one using the Search API.

In [1]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "df.csv"
CANDIDATE_PATH = PROJECT_ROOT / "data" / "candidate_tracks.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "recommendations.csv"


## 2. Load User Playlist

This is the user's known taste profile. Tracks from this playlist will be excluded from final recommendations.

In [3]:
playlist_df = pd.read_csv(DATA_PATH)

playlist_df["source"] = "playlist"
playlist_track_ids = set(playlist_df["track_id"].dropna())

print(playlist_df.shape)
playlist_df.head()

(238, 24)


,track_id,track_name,artist_names,artist_ids,main_artist_name,main_artist_id,album_id,album_name,release_date,duration_ms,duration_min,popularity,explicit,track_url,added_at,is_local,artist_name_from_api,main_artist_genres,artist_popularity,artist_followers,title_script,song_title_language,genre_language_tags,source
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,5lpy0FxmneKqFb4zeoiSHM,十八般武藝,2010-08-12,251146,4.185767,50,False,https://open.spotify.com/track/0VGPId3riSeBV5FTRn850F,2025-11-20T07:03:00Z,False,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",57,1192346,chinese,Chinese-script title,['Chinese-associated'],playlist
1,6aF2RVTPJSJCM1MpUjne5X,落葉歸根,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,1lxlQW182pklrfpD93faq1,改變自己,2007-07-01,315226,5.253767,51,False,https://open.spotify.com/track/6aF2RVTPJSJCM1MpUjne5X,2025-11-20T07:03:00Z,False,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",57,1192346,chinese,Chinese-script title,['Chinese-associated'],playlist
2,68RtalgSqaCbeZN42QeMS0,花田錯,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,34D46J9tIGCqAj3FeiEA9O,蓋世英雄,2005-12-30,227960,3.799333,50,False,https://open.spotify.com/track/68RtalgSqaCbeZN42QeMS0,2025-11-20T07:03:00Z,False,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",57,1192346,chinese,Chinese-script title,['Chinese-associated'],playlist
3,3agtg0x11wPvLIWkYR39nZ,Somewhere I Belong,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,4Gfnly5CzMJQqkUFfoHaP3,Meteora,2003-03-25,213933,3.565550,81,False,https://open.spotify.com/track/3agtg0x11wPvLIWkYR39nZ,2025-11-20T07:03:00Z,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34542635,latin,Latin-script title,['Western-associated'],playlist
4,57BrRMwf9LrcmuOsyGilwr,Crawling,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,6hPkbAV3ZXpGZBGUvL6jVM,Hybrid Theory (Bonus Edition),2000-10-24,208960,3.482667,84,False,https://open.spotify.com/track/57BrRMwf9LrcmuOsyGilwr,2025-11-20T07:03:00Z,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34542635,latin,Latin-script title,['Western-associated'],playlist


## 3. Optional: Build a Candidate Pool from Spotify Search

A recommendation model can only recommend from songs it has available as candidates. This section searches Spotify for tracks related to the user's top artists and genres, enriches those tracks with artist metadata, and saves them to `data/candidate_tracks.csv`.

Set credentials in a local `.env` file before running this section:

```text
CLIENT_ID=your_client_id
CLIENT_SECRET=your_client_secret
```

The helper below also supports the clearer names `SPOTIFY_CLIENT_ID` and `SPOTIFY_CLIENT_SECRET`.

Spotify's Search API supports track search with field filters such as artist, year, and genre. The notebook uses search because the recommendation endpoint and genre seed endpoint are deprecated in the current Spotify Web API reference.

In [4]:
def parse_list_like(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == "":
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return [part.strip() for part in str(value).split(",") if part.strip()]


def load_env_file(path=None):
    env_path = Path(path) if path is not None else PROJECT_ROOT / ".env"
    if not env_path.exists():
        return

    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


def get_spotify_token():
    load_env_file()
    client_id = os.getenv("SPOTIFY_CLIENT_ID") or os.getenv("SPOTIFY_CLIENT_ID")
    client_secret = os.getenv("SPOTIFY_CLIENT_SECRET") or os.getenv("SPOTIFY_CLIENT_SECRET")

    if not client_id or not client_secret:
        raise ValueError("Set CLIENT_ID and CLIENT_SECRET in .env before calling Spotify.")

    response = requests.post(
        "https://accounts.spotify.com/api/token",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def spotify_get(url, token, params=None, sleep_time=0.05):
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}"},
        params=params,
        timeout=30,
    )

    if response.status_code == 429:
        retry_after = int(response.headers.get("Retry-After", "1"))
        time.sleep(retry_after)
        return spotify_get(url, token, params=params, sleep_time=sleep_time)

    response.raise_for_status()
    time.sleep(sleep_time)
    return response.json()


def track_to_row(track):
    artists = track.get("artists", [])
    album = track.get("album", {})
    artist_names = [artist.get("name") for artist in artists if artist.get("name")]
    artist_ids = [artist.get("id") for artist in artists if artist.get("id")]

    return {
        "track_id": track.get("id"),
        "track_name": track.get("name"),
        "artist_names": ", ".join(artist_names),
        "artist_ids": ", ".join(artist_ids),
        "main_artist_name": artist_names[0] if artist_names else None,
        "main_artist_id": artist_ids[0] if artist_ids else None,
        "album_id": album.get("id"),
        "album_name": album.get("name"),
        "release_date": album.get("release_date"),
        "duration_ms": track.get("duration_ms"),
        "duration_min": track.get("duration_ms") / 60000 if track.get("duration_ms") else np.nan,
        "popularity": track.get("popularity"),
        "explicit": track.get("explicit"),
        "track_url": track.get("external_urls", {}).get("spotify"),
        "is_local": track.get("is_local", False),
    }


def top_playlist_genres(df, n=12):
    genre_counter = Counter()
    for genres in df["main_artist_genres"].map(parse_list_like):
        genre_counter.update(genres)
    return [genre for genre, _ in genre_counter.most_common(n)]


def search_tracks(query, token, market="US", limit=50):
    data = spotify_get(
        "https://api.spotify.com/v1/search",
        token,
        params={"q": query, "type": "track", "market": market, "limit": limit},
    )
    return data.get("tracks", {}).get("items", [])


def get_artist_metadata(artist_ids, token):
    rows = []
    for artist_id in sorted(set(artist_ids)):
        if not artist_id:
            continue
        artist = spotify_get(f"https://api.spotify.com/v1/artists/{artist_id}", token)
        rows.append({
            "main_artist_id": artist.get("id"),
            "artist_name_from_api": artist.get("name"),
            "main_artist_genres": artist.get("genres", []),
            "artist_popularity": artist.get("popularity"),
            "artist_followers": artist.get("followers", {}).get("total"),
        })
    return pd.DataFrame(rows)


def build_candidate_pool_from_spotify(playlist_df, market="US", per_query_limit=50):
    token = get_spotify_token()

    top_artists = playlist_df["main_artist_name"].value_counts().head(20).index.tolist()
    top_genres = top_playlist_genres(playlist_df, n=20)

    queries = []
    queries.extend([f'artist:"{artist}"' for artist in top_artists])
    queries.extend([f'genre:"{genre}"' for genre in top_genres])

    rows = []
    for query in queries:
        tracks = search_tracks(query, token, market=market, limit=per_query_limit)
        rows.extend(track_to_row(track) for track in tracks)

    candidates = pd.DataFrame(rows)
    candidates = candidates.dropna(subset=["track_id"]).drop_duplicates("track_id")
    candidates = candidates[~candidates["track_id"].isin(playlist_track_ids)].copy()

    artist_metadata = get_artist_metadata(candidates["main_artist_id"].dropna().unique(), token)
    candidates = candidates.merge(artist_metadata, on="main_artist_id", how="left")
    candidates["source"] = "candidate"

    CANDIDATE_PATH.parent.mkdir(exist_ok=True)
    candidates.to_csv(CANDIDATE_PATH, index=False)
    return candidates


# Run this only when you need to create or refresh the candidate pool.
# candidate_df = build_candidate_pool_from_spotify(playlist_df, market="US", per_query_limit=50)
# candidate_df.shape

## 4. Load Candidate Tracks

The final recommendations come from this candidate pool after removing tracks already present in the user's playlist.

In [5]:
if CANDIDATE_PATH.exists():
    candidate_df = pd.read_csv(CANDIDATE_PATH)
else:
    print("data/candidate_tracks.csv does not exist yet. Building it from Spotify Search...")
    candidate_df = build_candidate_pool_from_spotify(playlist_df, market="US", per_query_limit=50)

candidate_df["source"] = "candidate"
candidate_df = candidate_df.dropna(subset=["track_id"]).drop_duplicates("track_id")
candidate_df = candidate_df[~candidate_df["track_id"].isin(playlist_track_ids)].copy()

print(candidate_df.shape)
candidate_df.head()

(1292, 20)


,track_id,track_name,artist_names,artist_ids,main_artist_name,main_artist_id,album_id,album_name,release_date,duration_ms,duration_min,popularity,explicit,track_url,is_local,artist_name_from_api,main_artist_genres,artist_popularity,artist_followers,source
0,3xXBsjrbG1xQIm1xv1cKOt,One More Light,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,5Eevxp2BCbWq25ZdiXRwYd,One More Light,2017-05-19,255066,4.251100,79,False,https://open.spotify.com/track/3xXBsjrbG1xQIm1xv1cKOt,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34625451,candidate
1,1Vej0qeQ3ioKwpI6FUbRv1,Papercut,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,6hPkbAV3ZXpGZBGUvL6jVM,Hybrid Theory (Bonus Edition),2000-10-24,184866,3.081100,83,False,https://open.spotify.com/track/1Vej0qeQ3ioKwpI6FUbRv1,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34625451,candidate
2,1fLlRApgzxWweF1JTf8yM5,Given Up,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,2tlTBLz2w52rpGCLBGyGw6,Minutes to Midnight,2007-05-14,189293,3.154883,84,True,https://open.spotify.com/track/1fLlRApgzxWweF1JTf8yM5,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34625451,candidate
3,0sp00HSXkQyqTa6QqM0O8V,Leave Out All The Rest,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,2tlTBLz2w52rpGCLBGyGw6,Minutes to Midnight,2007-05-14,209306,3.488433,77,False,https://open.spotify.com/track/0sp00HSXkQyqTa6QqM0O8V,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34625451,candidate
4,60IkVf7UfQXmt5CwkpcX8a,From the Inside,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,4Gfnly5CzMJQqkUFfoHaP3,Meteora,2003-03-25,175506,2.925100,68,False,https://open.spotify.com/track/60IkVf7UfQXmt5CwkpcX8a,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34625451,candidate


## 5. Feature Engineering

The model uses metadata and artist-level features that are available for both playlist and candidate tracks:

- Numeric features: duration, track popularity, artist popularity, artist followers, release year.
- Binary/categorical features: explicit flag, title script, song title language.
- Multi-label features: artist genres and language-associated genre tags.

All features are fitted on the combined playlist + candidate set so both groups share the exact same columns.

In [6]:
def extract_release_year(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d{4}", str(value))
    return int(match.group()) if match else np.nan


def build_feature_matrix(df, top_genres=None, top_tags=None):
    model_df = df.copy()

    model_df["release_year"] = model_df["release_date"].map(extract_release_year)
    model_df["explicit"] = model_df["explicit"].astype(str).str.lower().isin(["true", "1", "yes"])
    model_df["artist_followers_log"] = np.log1p(pd.to_numeric(model_df.get("artist_followers"), errors="coerce"))

    numeric_cols = ["duration_min", "popularity", "artist_popularity", "artist_followers_log", "release_year"]
    numeric_features = model_df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    numeric_features = numeric_features.fillna(numeric_features.median())

    scaler = MinMaxScaler()
    numeric_features = pd.DataFrame(
        scaler.fit_transform(numeric_features),
        columns=numeric_cols,
        index=model_df.index,
    )

    categorical_cols = ["explicit", "title_script", "song_title_language"]
    for col in categorical_cols:
        if col not in model_df.columns:
            model_df[col] = "unknown"
    categorical_input = model_df[categorical_cols].fillna("unknown").astype(str)
    categorical_features = pd.get_dummies(categorical_input, prefix=categorical_cols)

    parsed_genres = model_df.get("main_artist_genres", pd.Series(index=model_df.index, dtype=object)).map(parse_list_like)
    parsed_tags = model_df.get("genre_language_tags", pd.Series(index=model_df.index, dtype=object)).map(parse_list_like)

    if top_genres is None:
        genre_counts = Counter(genre for genres in parsed_genres for genre in genres)
        top_genres = [genre for genre, _ in genre_counts.most_common(50)]

    if top_tags is None:
        tag_counts = Counter(tag for tags in parsed_tags for tag in tags)
        top_tags = [tag for tag, _ in tag_counts.most_common(20)]

    genre_features = pd.DataFrame(index=model_df.index)
    for genre in top_genres:
        genre_features[f"genre__{genre}"] = parsed_genres.map(lambda values, g=genre: int(g in values))

    tag_features = pd.DataFrame(index=model_df.index)
    for tag in top_tags:
        tag_features[f"tag__{tag}"] = parsed_tags.map(lambda values, t=tag: int(t in values))

    features = pd.concat([numeric_features, categorical_features, genre_features, tag_features], axis=1)
    return features, top_genres, top_tags


combined_df = pd.concat([playlist_df, candidate_df], ignore_index=True, sort=False)
features, top_genres, top_tags = build_feature_matrix(combined_df)

playlist_features = features.loc[combined_df["source"].eq("playlist")]
candidate_features = features.loc[combined_df["source"].eq("candidate")]

print(features.shape)
features.head()

(1530, 72)


,duration_min,popularity,artist_popularity,artist_followers_log,release_year,explicit_False,explicit_True,title_script_chinese,title_script_japanese,title_script_latin,title_script_mixed,title_script_unknown,song_title_language_Chinese-script title,song_title_language_Japanese-script title,song_title_language_Latin-script title,song_title_language_Mixed-script title,song_title_language_unknown,genre__mandopop,genre__c-pop,genre__taiwanese pop,genre__malay pop,genre__chinese r&b,genre__cantopop,genre__j-pop,genre__anime,genre__rap metal,genre__rock,genre__nu metal,genre__k-pop,genre__alternative metal,genre__gufeng,genre__pop,genre__tropical house,genre__country,genre__soft pop,genre__malay,genre__soft rock,genre__gothic metal,genre__malaysian pop,genre__classic rock,genre__yacht rock,genre__pop country,genre__post-grunge,genre__j-r&b,genre__christian,genre__acid jazz,genre__christian lo-fi,genre__christian r&b,genre__grunge,genre__edm,genre__rap,genre__alternative rock,genre__city pop,genre__j-rock,genre__funk rock,genre__chinese hip hop,genre__acoustic country,genre__uk r&b,genre__emo,genre__pop punk,genre__worship,genre__christian pop,genre__pop kreatif,genre__jazz funk,genre__enka,genre__christian hip hop,genre__folk metal,tag__Chinese-associated,tag__Western/Latin-title-associated,tag__Western-associated,tag__Korean-associated,tag__Japanese-associated
0,0.052069,0.510204,0.57,0.717045,0.741935,True,False,True,False,False,False,False,True,False,False,False,False,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,0.065655,0.520408,0.57,0.717045,0.693548,True,False,True,False,False,False,False,True,False,False,False,False,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,0.047153,0.510204,0.57,0.717045,0.661290,True,False,True,False,False,False,False,True,False,False,False,False,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0.044179,0.826531,0.89,0.911987,0.629032,True,False,False,False,True,False,False,False,False,True,False,False,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
4,0.043124,0.857143,0.89,0.911987,0.580645,True,False,False,False,True,False,False,False,False,True,False,False,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [7]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [8]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [9]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [10]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [11]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [12]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [13]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [14]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [15]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [16]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [17]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [18]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [19]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

In [20]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("data/df.csv")
CANDIDATE_PATH = Path("data/candidate_tracks.csv")
OUTPUT_PATH = Path("data/recommendations.csv")

## 6. Build Playlist Taste Profiles

The overall profile is the average feature vector across all songs in the user's playlist.

Weighted profiles can emphasize songs that are more recent additions or more popular. The default below keeps the baseline simple and transparent.

In [21]:
overall_profile = playlist_features.mean(axis=0).to_frame().T

similarities = cosine_similarity(candidate_features, overall_profile).ravel()

recommendations = candidate_df.copy()
recommendations["similarity_to_playlist"] = similarities
recommendations = recommendations.sort_values("similarity_to_playlist", ascending=False)

recommendations[[
    "track_name",
    "artist_names",
    "album_name",
    "release_date",
    "popularity",
    "artist_popularity",
    "main_artist_genres",
    "similarity_to_playlist",
    "track_url",
]].head(25)

,track_name,artist_names,album_name,release_date,popularity,artist_popularity,main_artist_genres,similarity_to_playlist,track_url
66,我對緣分小心翼翼 (劇集《逐玉》主題曲),JJ Lin,我對緣分小心翼翼 (劇集《逐玉》主題曲),2026-02-27,66,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.660313,https://open.spotify.com/track/6pCp8j2MmywUs8vygmqNQ1
62,交換餘生,JJ Lin,交換餘生,2020-09-16,61,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.653594,https://open.spotify.com/track/4daA20tBusVX29bUWgd8Dw
54,明日座標 (王者榮耀十週年版本主題曲),"JJ Lin, 王者荣耀",明日座標 (王者榮耀十週年版本主題曲),2025-10-09,49,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.653025,https://open.spotify.com/track/3gNdY7cdQckKDto9BP2dDW
43,願與愁,JJ Lin,願與愁,2023-04-14,51,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.652216,https://open.spotify.com/track/4VGvaHir2FQ4WMKn3GS9AS
36,裹著心的光,JJ Lin,裹著心的光,2021-06-09,55,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.652196,https://open.spotify.com/track/6ZpJxisQKOVQTthzmfN8TT
436,布拉格廣場 - JOLIN Version/ 星宇航空布拉格開航主題曲,JOLIN,布拉格廣場 (JOLIN Version/ 星宇航空布拉格開航主題曲),2026-02-25,57,61,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.650343,https://open.spotify.com/track/0HnESxAFR5lTHioAlI7wo3
63,黑夜問白天,JJ Lin,偉大的渺小,2017-12-28,59,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.650094,https://open.spotify.com/track/5KNh5YQgfduzV4028Cfh3J
40,願與愁,JJ Lin,重拾_快樂,2023-04-21,46,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649958,https://open.spotify.com/track/4ANuvW01ynBAu7rUS6sarM
67,不為誰而作的歌,JJ Lin,和自己對話,2015-12-25,62,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649217,https://open.spotify.com/track/1MGgqiQkHHiI9DarZulc12
37,將故事寫成我們,JJ Lin,將故事寫成我們,2019-09-20,52,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649189,https://open.spotify.com/track/3APYQVgmEDZ6sPbpLo7Hog


## 7. Recommendation Modes

The same feature matrix can support different recommendation experiences:

- `balanced`: most similar to the full playlist profile.
- `discovery`: similar to the playlist, but avoids artists already heavily represented in the playlist.
- `fresh`: similar to the playlist, with a small boost for newer releases.
- `popular`: similar to the playlist, with a small boost for track popularity.

In [22]:
playlist_artists = set(playlist_df["main_artist_id"].dropna())


def normalize_series(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.max() == values.min():
        return pd.Series(0.0, index=series.index)
    return (values - values.min()) / (values.max() - values.min())


def rank_recommendations(mode="balanced", n=25):
    ranked = recommendations.copy()
    ranked["score"] = ranked["similarity_to_playlist"]

    if mode == "discovery":
        known_artist_penalty = ranked["main_artist_id"].isin(playlist_artists).astype(float) * 0.08
        ranked["score"] = ranked["score"] - known_artist_penalty
    elif mode == "fresh":
        ranked["release_year"] = ranked["release_date"].map(extract_release_year)
        ranked["score"] = ranked["score"] + 0.05 * normalize_series(ranked["release_year"])
    elif mode == "popular":
        ranked["score"] = ranked["score"] + 0.05 * normalize_series(ranked["popularity"])
    elif mode != "balanced":
        raise ValueError("mode must be one of: balanced, discovery, fresh, popular")

    display_cols = [
        "track_name",
        "artist_names",
        "album_name",
        "release_date",
        "popularity",
        "main_artist_genres",
        "similarity_to_playlist",
        "score",
        "track_url",
    ]
    return ranked.sort_values("score", ascending=False).head(n)[display_cols]


rank_recommendations(mode="balanced", n=20)

,track_name,artist_names,album_name,release_date,popularity,main_artist_genres,similarity_to_playlist,score,track_url
66,我對緣分小心翼翼 (劇集《逐玉》主題曲),JJ Lin,我對緣分小心翼翼 (劇集《逐玉》主題曲),2026-02-27,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.660313,0.660313,https://open.spotify.com/track/6pCp8j2MmywUs8vygmqNQ1
62,交換餘生,JJ Lin,交換餘生,2020-09-16,61,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.653594,0.653594,https://open.spotify.com/track/4daA20tBusVX29bUWgd8Dw
54,明日座標 (王者榮耀十週年版本主題曲),"JJ Lin, 王者荣耀",明日座標 (王者榮耀十週年版本主題曲),2025-10-09,49,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.653025,0.653025,https://open.spotify.com/track/3gNdY7cdQckKDto9BP2dDW
43,願與愁,JJ Lin,願與愁,2023-04-14,51,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.652216,0.652216,https://open.spotify.com/track/4VGvaHir2FQ4WMKn3GS9AS
36,裹著心的光,JJ Lin,裹著心的光,2021-06-09,55,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.652196,0.652196,https://open.spotify.com/track/6ZpJxisQKOVQTthzmfN8TT
436,布拉格廣場 - JOLIN Version/ 星宇航空布拉格開航主題曲,JOLIN,布拉格廣場 (JOLIN Version/ 星宇航空布拉格開航主題曲),2026-02-25,57,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.650343,0.650343,https://open.spotify.com/track/0HnESxAFR5lTHioAlI7wo3
63,黑夜問白天,JJ Lin,偉大的渺小,2017-12-28,59,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.650094,0.650094,https://open.spotify.com/track/5KNh5YQgfduzV4028Cfh3J
40,願與愁,JJ Lin,重拾_快樂,2023-04-21,46,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649958,0.649958,https://open.spotify.com/track/4ANuvW01ynBAu7rUS6sarM
67,不為誰而作的歌,JJ Lin,和自己對話,2015-12-25,62,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649217,0.649217,https://open.spotify.com/track/1MGgqiQkHHiI9DarZulc12
37,將故事寫成我們,JJ Lin,將故事寫成我們,2019-09-20,52,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649189,0.649189,https://open.spotify.com/track/3APYQVgmEDZ6sPbpLo7Hog


In [23]:
rank_recommendations(mode="discovery", n=20)

,track_name,artist_names,album_name,release_date,popularity,main_artist_genres,similarity_to_playlist,score,track_url
1185,The Fate of Ophelia,Taylor Swift,The Life of a Showgirl,2025-10-03,96,[],0.635855,0.635855,https://open.spotify.com/track/53iuhJlwXhSER5J2IYYv1W
1184,Opalite,Taylor Swift,The Life of a Showgirl,2025-10-03,91,[],0.635715,0.635715,https://open.spotify.com/track/3yWuTOYDztXjZxdE2cIRUa
1189,I Just Might,Bruno Mars,The Romantic,2026-02-27,90,[],0.634515,0.634515,https://open.spotify.com/track/6f68Ac5tt5BQOpwr0BV9oN
1183,BIRDS OF A FEATHER,Billie Eilish,HIT ME HARD AND SOFT,2024-05-17,95,[],0.633337,0.633337,https://open.spotify.com/track/6dOtVTDdiauQNBQEDOtlAB
1174,drop dead,Olivia Rodrigo,drop dead,2026-04-17,93,[],0.632289,0.632289,https://open.spotify.com/track/6gkbtMtioHgtyGjrMel6ei
1176,drop dead,Olivia Rodrigo,drop dead (taken that eurostar to france video),2026-04-16,89,[],0.632120,0.632120,https://open.spotify.com/track/7Hc6qcJG4NtyZgbNvQyd8U
1191,American Girls,Harry Styles,"Kiss All The Time. Disco, Occasionally.",2026-03-06,93,[],0.631235,0.631235,https://open.spotify.com/track/7gtG45ieyQzKtNKobfLd49
621,Guy For That (Feat. Luke Combs),"Post Malone, Luke Combs",F-1 Trillion,2024-08-15,76,[],0.628200,0.628200,https://open.spotify.com/track/6StwwqB84sJeLr7tZDTxEX
620,Pour Me A Drink (Feat. Blake Shelton),"Post Malone, Blake Shelton",F-1 Trillion,2024-08-15,75,[],0.628048,0.628048,https://open.spotify.com/track/0mNzElhEofvgMWAJoOA4q9
993,Take Me to the Beach (feat. Ado),"Imagine Dragons, Ado",Take Me to the Beach (feat. Ado),2024-12-16,73,[],0.627801,0.627801,https://open.spotify.com/track/4n7nwDqrdx3qmJvEACOMrN


## 8. Lightweight Validation

Recommendation quality is hard to measure without real user feedback. As a sanity check, this leave-one-out style evaluation asks whether a held-out playlist track looks similar to the rest of the playlist.

This does not prove the recommendations are good, but it checks whether the feature space captures the playlist's internal structure.

In [24]:
def playlist_holdout_similarity(playlist_features, holdout_fraction=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    n_holdout = max(1, int(len(playlist_features) * holdout_fraction))
    holdout_idx = rng.choice(playlist_features.index.to_numpy(), size=n_holdout, replace=False)

    train_features = playlist_features.drop(index=holdout_idx)
    holdout_features = playlist_features.loc[holdout_idx]

    train_profile = train_features.mean(axis=0).to_frame().T
    holdout_similarity = cosine_similarity(holdout_features, train_profile).ravel()

    random_similarity = cosine_similarity(
        candidate_features.sample(n=min(n_holdout, len(candidate_features)), random_state=random_state),
        train_profile,
    ).ravel()

    return pd.DataFrame({
        "group": ["held_out_playlist"] * len(holdout_similarity) + ["random_candidate"] * len(random_similarity),
        "similarity": np.concatenate([holdout_similarity, random_similarity]),
    })


validation_df = playlist_holdout_similarity(playlist_features)
validation_df.groupby("group")["similarity"].describe()

,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
held_out_playlist,47.0,0.753486,0.101031,0.419387,0.711778,0.766185,0.826145,0.86878
random_candidate,47.0,0.542958,0.075009,0.397394,0.472499,0.557020,0.613651,0.65245


## 9. Export Final Recommendations

Save a ranked recommendation table for later use in a report or app.

In [25]:
final_recommendations = rank_recommendations(mode="balanced", n=100)
OUTPUT_PATH.parent.mkdir(exist_ok=True)
final_recommendations.to_csv(OUTPUT_PATH, index=False)

print(f"Saved recommendations to {OUTPUT_PATH}")
final_recommendations.head(20)

Saved recommendations to data/recommendations.csv


,track_name,artist_names,album_name,release_date,popularity,main_artist_genres,similarity_to_playlist,score,track_url
66,我對緣分小心翼翼 (劇集《逐玉》主題曲),JJ Lin,我對緣分小心翼翼 (劇集《逐玉》主題曲),2026-02-27,66,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.660313,0.660313,https://open.spotify.com/track/6pCp8j2MmywUs8vygmqNQ1
62,交換餘生,JJ Lin,交換餘生,2020-09-16,61,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.653594,0.653594,https://open.spotify.com/track/4daA20tBusVX29bUWgd8Dw
54,明日座標 (王者榮耀十週年版本主題曲),"JJ Lin, 王者荣耀",明日座標 (王者榮耀十週年版本主題曲),2025-10-09,49,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.653025,0.653025,https://open.spotify.com/track/3gNdY7cdQckKDto9BP2dDW
43,願與愁,JJ Lin,願與愁,2023-04-14,51,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.652216,0.652216,https://open.spotify.com/track/4VGvaHir2FQ4WMKn3GS9AS
36,裹著心的光,JJ Lin,裹著心的光,2021-06-09,55,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.652196,0.652196,https://open.spotify.com/track/6ZpJxisQKOVQTthzmfN8TT
436,布拉格廣場 - JOLIN Version/ 星宇航空布拉格開航主題曲,JOLIN,布拉格廣場 (JOLIN Version/ 星宇航空布拉格開航主題曲),2026-02-25,57,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.650343,0.650343,https://open.spotify.com/track/0HnESxAFR5lTHioAlI7wo3
63,黑夜問白天,JJ Lin,偉大的渺小,2017-12-28,59,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.650094,0.650094,https://open.spotify.com/track/5KNh5YQgfduzV4028Cfh3J
40,願與愁,JJ Lin,重拾_快樂,2023-04-21,46,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649958,0.649958,https://open.spotify.com/track/4ANuvW01ynBAu7rUS6sarM
67,不為誰而作的歌,JJ Lin,和自己對話,2015-12-25,62,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649217,0.649217,https://open.spotify.com/track/1MGgqiQkHHiI9DarZulc12
37,將故事寫成我們,JJ Lin,將故事寫成我們,2019-09-20,52,"['mandopop', 'c-pop', 'taiwanese pop', 'malay pop']",0.649189,0.649189,https://open.spotify.com/track/3APYQVgmEDZ6sPbpLo7Hog


## 10. Modelling Takeaways

- This is a content-based recommender, which fits the project because the data is centered on one user's playlist rather than many users' listening histories.
- The model recommends only from the candidate pool, so candidate generation is as important as the similarity model.
- The similarity score is interpretable because it is based on genre, language, artist metadata, release year, duration, and popularity.
- Future improvements could add user feedback, audio features if available, playlist section-specific profiles, or an app interface where the user can choose between balanced, discovery, fresh, and popular modes.